In [ ]:

import pandas as pd
import pathlib
import glob
import os
import numpy as np
import warnings
import gc  # Added for memory management
import pyanalib.split_df_helpers as splh
from tables import NaturalNameWarning

# 1. Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=NaturalNameWarning)
from tqdm import tqdm # Standard for progress bars in SBND environments
def consolidate_buffered(folder_path, output_name, split_margin_gb=1.0, num_splits=1, shift_ntuple_index=False):
    """
    Consolidates files with optional global ntuple index shifting.
    """
    out_dir = pathlib.Path("/exp/sbnd/data/users/lpelegri/cafpyana_data")
    out_dir.mkdir(parents=True, exist_ok=True)
    output_file = out_dir / f"{output_name}.df"

    input_folder = pathlib.Path(folder_path)
    files = sorted(glob.glob(str(input_folder / f"{output_name}_*.df")))
    
    if not files:
        print(f"No files found for {output_name}")
        return

    # --- ANNOUNCE START ---
    print(f"\n{'='*60}")
    print(f"COMBINING: {output_name}")
    print(f"TARGET:    {output_file}")
    print(f"SHIFT INDEX: {shift_ntuple_index}")
    print(f"TOTAL FILES TO PROCESS: {len(files)}")
    print(f"{'='*60}\n")

    # 1. Discover keys
    with pd.HDFStore(files[0], mode='r') as store:
        keys2load = [k.lstrip('/').rsplit('_', 1)[0] for k in store.keys() if 'split' not in k]
        keys2load = sorted(list(set(keys2load)))

    df_buffers = {k: [] for k in keys2load}
    size_counters = {k: 0 for k in keys2load}
    k_idx = 0 
    total_ntuples_seen = 0 

    with pd.HDFStore(output_file, mode='w') as hdf_out:
        pbar = tqdm(files, desc="Consolidating", unit="file")
        
        for file_path in pbar:
            if k_idx >= num_splits:
                pbar.write(f"Reached limit of {num_splits} splits. Stopping.")
                break
            
            file_data = splh.load_dfs(file_path, keys2load)
            
            # --- INDEX MAPPING LOGIC ---
            if shift_ntuple_index:
                # Calculate unique IDs only if we need to shift
                unique_ids_in_file = set()
                for df in file_data.values():
                    if df is not None and not df.empty:
                        unique_ids_in_file.update(df.index.get_level_values("__ntuple").unique())
                
                sorted_old_ids = sorted(list(unique_ids_in_file))
                num_unique_ntuples = len(sorted_old_ids)

                if num_unique_ntuples > 0:
                    mapping = {old_id: i + total_ntuples_seen for i, old_id in enumerate(sorted_old_ids)}
                    
                    for k in keys2load:
                        df = file_data[k]
                        if df is not None and not df.empty:
                            # Apply the mapping to the index
                            old_indices = df.index.get_level_values("__ntuple")
                            new_indices = old_indices.map(mapping)
                            df.index = df.index.set_levels(new_indices.unique(), level="__ntuple")
                    
                    total_ntuples_seen += num_unique_ntuples

            # --- BUFFERING ---
            for k in keys2load:
                df = file_data[k]
                if df is not None and not df.empty:
                    size_gb = df.memory_usage(deep=True).sum() / (1024**3)
                    size_counters[k] += size_gb
                    df_buffers[k].append(df)
            
            # --- FLUSH LOGIC ---
            current_total_buffer_gb = sum(size_counters.values())
            if current_total_buffer_gb >= split_margin_gb:
                pbar.write(f"--- Buffer limit reached ({current_total_buffer_gb:.3f} GB). Flushing to split_{k_idx} ---")
                for k in keys2load:
                    if df_buffers[k]:
                        concat_df = pd.concat(df_buffers[k], ignore_index=False)
                        hdf_out.put(key=f"{k}_{k_idx}", value=concat_df, format="fixed")
                        df_buffers[k] = [] 
                        size_counters[k] = 0
                k_idx += 1
                gc.collect()

                if k_idx >= num_splits:
                    break
                
            del file_data
            gc.collect()

        # Final Flush for remaining data
        if k_idx < num_splits and any(len(b) > 0 for b in df_buffers.values()):
            print(f"--- Final Flush: split_{k_idx} ---")
            for k in keys2load:
                if df_buffers[k]:
                    concat_df = pd.concat(df_buffers[k], ignore_index=False)
                    hdf_out.put(key=f"{k}_{k_idx}", value=concat_df, format="fixed")
            k_idx += 1
            gc.collect()

        hdf_out.put(key="split", value=pd.DataFrame({"n_split": [k_idx]}), format="fixed")

    print(f"\nConsolidation finished. Total splits: {k_idx}")
    
    # Cleanup block
    print("Deleting temporary files...")
    for file_path in files:
        try:
            os.remove(file_path)
        except OSError:
            pass 
    print("All Clean!")
    return output_file
    
# --- Execution ---
# Dictionary mapping file names to their shift_ntuple_index status
'''
file_config = {
    "cc1pi_5e18_CV": False,
    "mc_GIBUU_gen1": False,
    "cc1pi_data_rollingdev_bnblight": False, # Usually False for data
    "cc1pi_data_offbeamlight": False,        # Usually False for data
    "cc1pi_1e20_lowE_CV": True,
    "cc1pi_5e18_in_time_cosmics": False,
    "cc1pi_5e18_CV_extra_syst": False,
    "cc1pi_extended_syst": False,
    "cc1pi_SystVarsCV": False,
    "cc1pi_ccalp": False,
    "cc1pi_alphap": False,
    "cc1pi_ccalm": False,
    "cc1pi_alpham": False,
    "cc1pi_betam": False,
    "cc1pi_rm": False,
    "cc1pi_rp": False,
    "cc1pi_betap": False,
    "cc1pi_wiremod_YZ": False,
    "cc1pi_wiremod_XZ_thetaXW": False,
    "cc1pi_0xSCE": False,
    "cc1pi_2xSCE": False,
    "cc1pi_PMTHighNoise": False,
    "cc1pi_PMTGainFluct": False,
    "cc1pi_PMTLowEff": False
}
'''
file_config = {
    "cc1pi_1e20_lowE_CV": False, # Usually False for data
}

SOURCE = "/exp/sbnd/data/users/lpelegri/cafpyana_data_transfer_folder"

for file_name, should_shift in file_config.items():
    print(f"\n>>> Processing {file_name} (Shift Index: {should_shift})")
    
    consolidate_buffered(
        SOURCE, 
        file_name, 
        split_margin_gb=1, 
        num_splits=1000, 
        shift_ntuple_index=True
    )

In [ ]:

def consolidate_buffered_big_files(folder_path, output_name, split_margin_gb=1.0, num_splits=1, shift_ntuple_index=True):
    """
    Consolidates files with optional global ntuple index shifting.
    """
    out_dir = pathlib.Path("/exp/sbnd/data/users/lpelegri/cafpyana_data")
    out_dir.mkdir(parents=True, exist_ok=True)
    output_file = out_dir / f"{output_name}.df"

    input_folder = pathlib.Path(folder_path)
    files = sorted(glob.glob(str(input_folder / f"{output_name}_*.df")))
    
    if not files:
        print(f"No files found for {output_name}")
        return

    # --- ANNOUNCE START ---
    print(f"\n{'='*60}")
    print(f"COMBINING: {output_name}")
    print(f"TARGET:    {output_file}")
    print(f"SHIFT INDEX: {shift_ntuple_index}")
    print(f"TOTAL FILES TO PROCESS: {len(files)}")
    print(f"{'='*60}\n")

    # 1. Discover keys
    with pd.HDFStore(files[0], mode='r') as store:
        keys2load = [k.lstrip('/').rsplit('_', 1)[0] for k in store.keys() if 'split' not in k]
        keys2load = sorted(list(set(keys2load)))

    df_buffers = {k: [] for k in keys2load}
    size_counters = {k: 0 for k in keys2load}
    k_idx = 0 
    total_ntuples_seen = 0 

    with pd.HDFStore(output_file, mode='w') as hdf_out:
        pbar = tqdm(files, desc="Consolidating", unit="file")
        
        for file_path in pbar:
            if k_idx >= num_splits:
                pbar.write(f"Reached limit of {num_splits} splits. Stopping.")
                break
            
            file_data = splh.load_dfs(file_path, keys2load)
            
            # --- INDEX MAPPING LOGIC ---
            if shift_ntuple_index:
                # Calculate unique IDs only if we need to shift
                unique_ids_in_file = set()
                for df in file_data.values():
                    if df is not None and not df.empty:
                        unique_ids_in_file.update(df.index.get_level_values("__ntuple").unique())
                
                sorted_old_ids = sorted(list(unique_ids_in_file))
                num_unique_ntuples = len(sorted_old_ids)

                if num_unique_ntuples > 0:
                    mapping = {old_id: i + total_ntuples_seen for i, old_id in enumerate(sorted_old_ids)}
                    
                    for k in keys2load:
                        df = file_data[k]
                        if df is not None and not df.empty:
                            # Apply the mapping to the index
                            old_indices = df.index.get_level_values("__ntuple")
                            new_indices = old_indices.map(mapping)
                            df.index = df.index.set_levels(new_indices.unique(), level="__ntuple")
                    
                    total_ntuples_seen += num_unique_ntuples

            # --- BUFFERING ---
            for k in keys2load:
                df = file_data[k]
                if df is not None and not df.empty:
                    size_gb = df.memory_usage(deep=True).sum() / (1024**3)
                    size_counters[k] += size_gb
                    df_buffers[k].append(df)
            
            # --- FLUSH LOGIC ---
            current_total_buffer_gb = sum(size_counters.values())
            if current_total_buffer_gb >= split_margin_gb:
                pbar.write(f"--- Buffer limit reached ({current_total_buffer_gb:.3f} GB). Flushing to split_{k_idx} ---")
                for k in keys2load:
                    if df_buffers[k]:
                        concat_df = pd.concat(df_buffers[k], ignore_index=False)
                        hdf_out.put(key=f"{k}_{k_idx}", value=concat_df, format="fixed")
                        df_buffers[k] = [] 
                        size_counters[k] = 0
                k_idx += 1
                gc.collect()

                if k_idx >= num_splits:
                    break
                
            del file_data
            gc.collect()

        # Final Flush for remaining data
        if k_idx < num_splits and any(len(b) > 0 for b in df_buffers.values()):
            print(f"--- Final Flush: split_{k_idx} ---")
            for k in keys2load:
                if df_buffers[k]:
                    concat_df = pd.concat(df_buffers[k], ignore_index=False)
                    hdf_out.put(key=f"{k}_{k_idx}", value=concat_df, format="fixed")
            k_idx += 1
            gc.collect()

        hdf_out.put(key="split", value=pd.DataFrame({"n_split": [k_idx]}), format="fixed")

    print(f"\nConsolidation finished. Total splits: {k_idx}")
    
    # Cleanup block
    print("Deleting temporary files...")
    '''
    for file_path in files:
        try:
            os.remove(file_path)
        except OSError:
            pass 
    print("All Clean!")
    '''
    return output_file


In [ ]:
file_list = [
    "mc_ar23p_extended_syst",
]


SOURCE = "/exp/sbnd/data/users/lpelegri/cafpyana_data_transfer_folder"

for file in file_list:
    consolidate_buffered_big_files(SOURCE, file, split_margin_gb=1, num_splits=10, shift_ntuple_index = True)